In [1]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

from models.CDC import keyframe_compressor as compress_modules
import torch
import torch.nn as nn
import importlib
import models
from torch.utils.data import DataLoader, ConcatDataset

importlib.reload(compress_modules)
from models.CDC import keyframe_compressor as compress_modules
from models.video_diffusion_interpo import Unet3D, GaussianDiffusion

importlib.reload(models.video_diffusion_interpo)
from models.video_diffusion_interpo import Unet3D, GaussianDiffusion
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device, torch.cuda.device_count()

/apps/pytorch/2.2.0/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(device(type='cuda'), 1)

In [2]:
def remove_module_prefix(state_dict):
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace("module.", "")  # remove 'module.' prefix
        new_state_dict[new_key] = v
    return new_state_dict


def load_model(pretrain_vae, pretrain_diffusion, time_steps=1000, sr=False, multi_gpu=False, device='cuda:0'):
    if not sr:
        keyframe_model = compress_modules.ResnetCompressor(
            dim=16,
            dim_mults=[1, 2, 3, 4],
            reverse_dim_mults=[4, 3, 2, 1],
            hyper_dims_mults=[4, 4, 4],
            channels=1,
            out_channels=1
        )
    else:
        keyframe_model = compress_modules.CompressorSR(
            dim=16,
            dim_mults=[1, 2, 3, 4],
            reverse_dim_mults=[4, 3, 2],
            hyper_dims_mults=[4, 4, 4],
            channels=1,
            out_channels=1,
            sr_dim=16
        )
        print("loading VAE2D model with SR")

    keyframe_model.load_state_dict(torch.load(pretrain_vae, map_location=device))
    keyframe_model = keyframe_model.to(device).eval()

    if multi_gpu:
        keyframe_model = torch.nn.DataParallel(keyframe_model)

    model = Unet3D(
        dim=64,
        out_dim=64,
        channels=64,
        dim_mults=(1, 2, 4, 8),
        use_bert_text_cond=False
    )

    diffusion = GaussianDiffusion(
        model,
        image_size=16,
        num_frames=10,
        channels=64,
        timesteps=time_steps,
        loss_type='l2'
    ).to(device).eval()

    diffusion.load_state_dict(torch.load(pretrain_diffusion, map_location=device)['ema'])

    if multi_gpu:
        diffusion = torch.nn.DataParallel(diffusion)

    return keyframe_model, diffusion


In [5]:
from models.CDC import compress_modules3d_mid_SR as compress_modules
model = compress_modules.CompressorMix( dim= 16,
                                        dim_mults=[1,2,3,4],
                                        reverse_dim_mults=[4,3,2],
                                        hyper_dims_mults=[4,4,4],
                                        channels = 1,
                                        out_channels = 1,
                                        d3=True,
                                        sr_dim = 64).cuda()

Loading BCRN model


In [7]:
import torch
import time

shape = [128, 1, 8, 256, 256]
data = torch.zeros(shape).cuda()
n_iter = 10

with torch.no_grad():
    # Warm up run (important for GPU timing accuracy)
    for _ in range(3):
        result = model(data, return_time=True)

    total_encoding = 0.0
    total_decoding = 0.0

    for _ in range(n_iter):
        result = model(data, return_time=True)
        total_encoding += result["encoding_time"]
        total_decoding += result["decoding_time"]

# compute total data size in GB
bytes_per_tensor = torch.prod(torch.tensor(shape)) * 4  # 4 bytes for float32
total_gb = (bytes_per_tensor.item() * n_iter) / (1024 ** 3)

# compute throughput in GB/s
encoding_throughput = total_gb / total_encoding
decoding_throughput = total_gb / total_decoding

print(f"Encoding speed: {encoding_throughput:.3f} GB/s")
print(f"Decoding speed: {decoding_throughput:.3f} GB/s")
        
    

Encoding speed: 0.943 GB/s
Decoding speed: 0.652 GB/s
